# Reproduce `monoT5_CT` (h2oloo's KZ-tuned reranker) — DOCUMENTED NEGATIVE RESULT

**Outcome: this reproduction does not succeed, and monoT5_CT is not used in the ctmatch pipeline.**
h2oloo's winning TREC22 reranker is `monoT5-3B-MED` fine-tuned on Koopman-Zuccon (KZ), reported at
TREC21 nDCG@10 0.7118. Their KZ-tuned checkpoint is not public. We have KZ and the published recipe, so
we attempted a faithful reproduction. It fails the same way every time:

- **Base model** (`castorini/monot5-3b-med-msmarco`, no KZ fine-tune): TREC21 judged-pool NDCG@10 = **0.4491**.
- **Every fine-tune** — five full fine-tunes (TF-IDF → BM25 → monoT5_MED hard negatives; full-text →
  sliding-window MaxP training) **and** the rank-16 LoRA below — collapses TREC21 to **~0.14–0.20 (at random)**.
- **Mechanism: KZ→TREC21 transfer failure, not a tuning bug.** KZ is 59 one-liner topics (~10 tokens)
  matched on the condition field; TREC21 is 150-word clinical narratives with multi-condition eligibility
  reasoning. The fine-tune learns KZ's pattern and does not transfer; with base weights *frozen* (LoRA),
  the adapter delta still shifts the output distribution off the base model on out-of-distribution TREC21
  input, so the TREC21 canary (below) collapses within ~50 steps.

**Conclusion.** monoT5_CT does not reproduce without h2oloo's non-public monoT5_MED-scored-negative
pipeline and exact checkpoint. The base model (0.4491) is what we use, as an *optional* ensemble feature.
Full run history (Runs 1–6, with per-run diagnostics) is in `docs/deep_dive_outline.md` §7e.

---
**Recipe attempted (Pradeep et al. SIGIR 2022, §3.2–3.3):** base `castorini/monot5-3b-med-msmarco`;
KZ positives (rel≥1) / negatives (0); monoT5 `Query: … Document: … Relevant:` → `true`/`false`;
multi-field templates; 3–4 hard + 1 weak negative per positive; here LoRA (r=16) in place of the full
fine-tune to test whether freezing base weights avoids the collapse (it does not). Documented deviation
from h2oloo: we approximate their monoT5_MED sliding-window segment selection with MaxP windows.


In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q transformers accelerate datasets sentencepiece requests tqdm peft
!pip install -q -U torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DATA_ROOT  = '/content/drive/MyDrive/ct_data23'
KZ_ROOT    = f'{DATA_ROOT}/evaluation/kz_data'
KZ_QRELS   = f'{KZ_ROOT}/qrels-clinical_trials.txt'
KZ_TOPICS  = f'{KZ_ROOT}/topics-2014_2015-description.topics'
KZ_FIELDS  = f'{DATA_ROOT}/kz_trial_fields.jsonl'
BASE_MODEL = 'castorini/monot5-3b-med-msmarco'
OUT_DIR    = f'{DATA_ROOT}/monot5_ct'

# h2oloo recipe: 1k steps, batch 128, LR 1e-3, Adafactor.
# LoRA fine-tuning: base weights frozen → no catastrophic forgetting.
# LR=1e-3 is fine for adapter weights (they start from zero).
# MICRO=8: T5-3B d_ff=16384 → OOM at MICRO=16 without GC.
STEPS, BATCH, MICRO, LR = 1000, 128, 8, 1e-3
MAX_LEN, DOC_CHARS = 512, 1400

# LoRA: rank-16 adapters on all attention projections.
# Trainable params ~14M vs 3B full model (~0.5%); second moments ~28MB vs 24GB for Adam.
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05

os.makedirs(OUT_DIR, exist_ok=True)
print('config set')

In [ ]:
import json, requests, time
from ctmatch.evaluation.eval_utils import get_kz_topic2text  # same parser the evaluator uses

# KZ topics + qrels
topic2text = get_kz_topic2text(KZ_TOPICS)
judg = {}
ncts = set()
for l in open(KZ_QRELS):
    t, _, d, r = l.split(); judg.setdefault(t, {})[d] = int(r); ncts.add(d)
print(f'KZ: {len(topic2text)} topics, {sum(len(v) for v in judg.values())} judgments, {len(ncts)} trials')

# fetch trial fields (title/condition/summary/detailed/eligibility) for judged NCT IDs (cache)
have = set()
if os.path.exists(KZ_FIELDS):
    have = {json.loads(l)['nct_id'] for l in open(KZ_FIELDS)}
todo = [n for n in ncts if n not in have]
API = 'https://clinicaltrials.gov/api/v2/studies'
FLD = ','.join(['protocolSection.identificationModule.nctId','protocolSection.identificationModule.briefTitle',
  'protocolSection.identificationModule.officialTitle','protocolSection.conditionsModule.conditions',
  'protocolSection.descriptionModule.briefSummary','protocolSection.descriptionModule.detailedDescription',
  'protocolSection.eligibilityModule.eligibilityCriteria'])
from tqdm.auto import tqdm
with open(KZ_FIELDS, 'a') as f:
    for i in tqdm(range(0, len(todo), 100), desc='fetch KZ trials'):
        b = todo[i:i+100]
        try:
            r = requests.get(API, params={'filter.ids': ','.join(b), 'pageSize': len(b), 'fields': FLD}, timeout=30)
            for s in r.json().get('studies', []):
                ps = s.get('protocolSection', {})
                idm, cm, dm, em = (ps.get(k, {}) for k in ['identificationModule','conditionsModule','descriptionModule','eligibilityModule'])
                f.write(json.dumps({'nct_id': idm.get('nctId',''),
                    'title': idm.get('officialTitle') or idm.get('briefTitle',''),
                    'condition': (cm.get('conditions') or [''])[0],
                    'summary': dm.get('briefSummary',''), 'detailed': dm.get('detailedDescription',''),
                    'eligibility': em.get('eligibilityCriteria','')}) + '\n')
        except Exception as e:
            print('batch failed', e)
        time.sleep(0.1)
fields = {r['nct_id']: r for r in map(json.loads, open(KZ_FIELDS))}
print(f'trial fields for {len(fields)} trials')

In [ ]:
# ── MaxP pre-selection + monoT5_MED hard-negative scoring (h2oloo §3.3) ─────
# Two things happen here with one base-model pass:
#   1. For each (topic, trial) pair, score all eligibility windows and cache
#      the best window (MaxP) as the training document.
#   2. Cache the MaxP score — used to RANK hard negatives.
#      h2oloo selects hard negatives as trials the BASE MODEL scores highest
#      but are labeled negative (the model's own false positives). BM25 negatives
#      are lexically similar but often already scored low by the base model →
#      weak gradient → forgetting dominates. monoT5_MED negatives are targeted.

MAXP_CACHE   = f'{DATA_ROOT}/kz_maxp_selections.jsonl'
WINDOW_CHARS = 600   # ≈ 6 short eligibility sentences / ~150 tokens
STRIDE_CHARS = 300   # 50% overlap

def eligibility_windows(elig_text):
    if not elig_text:
        return ['']
    wins, pos = [], 0
    while pos < len(elig_text):
        wins.append(elig_text[pos:pos + WINDOW_CHARS])
        if pos + WINDOW_CHARS >= len(elig_text):
            break
        pos += STRIDE_CHARS
    return wins or ['']

# Check whether an existing cache has the maxp_score field (v2 format).
# If it was written by an older run (no score), delete and regenerate.
cache_stale = False
if os.path.exists(MAXP_CACHE):
    with open(MAXP_CACHE) as _f:
        _first = _f.readline()
    if _first and 'maxp_score' not in _first:
        print('Cache is old format (no maxp_score) — deleting and regenerating.')
        os.remove(MAXP_CACHE)
        cache_stale = True

if os.path.exists(MAXP_CACHE):
    maxp_windows, maxp_scores = {}, {}
    for line in open(MAXP_CACHE):
        r = json.loads(line)
        key = (r['topic_id'], r['nct_id'])
        maxp_windows[key] = r['elig_window']
        maxp_scores[key]  = r['maxp_score']
    print(f'Loaded MaxP cache: {len(maxp_windows):,} (topic, trial) pairs')
else:
    import torch
    from transformers import T5Tokenizer, T5ForConditionalGeneration
    print('Loading base model for MaxP selection + hard-negative scoring...')
    mp_tok   = T5Tokenizer.from_pretrained(BASE_MODEL)
    mp_model = T5ForConditionalGeneration.from_pretrained(
        BASE_MODEL, torch_dtype=torch.float16, device_map='auto').eval()
    MP_TRUE  = mp_tok('true',  add_special_tokens=False).input_ids[0]
    MP_FALSE = mp_tok('false', add_special_tokens=False).input_ids[0]

    def _score_windows(topic_text, win_strs, batch=16):
        inputs = [f'Query: {topic_text} Document: {w} Relevant:' for w in win_strs]
        scores = []
        for i in range(0, len(inputs), batch):
            enc = mp_tok(inputs[i:i+batch], return_tensors='pt', padding=True,
                         truncation=True, max_length=MAX_LEN).to(mp_model.device)
            dec = torch.zeros((enc['input_ids'].shape[0], 1), dtype=torch.long,
                              device=mp_model.device)
            with torch.no_grad():
                out = mp_model(**enc, decoder_input_ids=dec).logits[:, 0, :]
            lp = torch.log_softmax(out.float(), dim=-1)
            scores.extend((lp[:, MP_TRUE] - lp[:, MP_FALSE]).cpu().tolist())
        return scores

    maxp_windows, maxp_scores = {}, {}
    with open(MAXP_CACHE, 'w') as cf:
        for tid, d2r in tqdm(judg.items(), desc='MaxP + score'):
            topic_text = topic2text.get(tid, '')
            if not topic_text:
                continue
            for nct in d2r:
                r = fields.get(nct)
                if not r:
                    continue
                elig  = r.get('eligibility', '') or ''
                title = r.get('title', '') or ''
                cond  = r.get('condition', '') or ''
                wins = eligibility_windows(elig)
                win_strs = [
                    f"title: {title} condition: {cond} eligibility: {w}"[:DOC_CHARS]
                    for w in wins
                ]
                scores = _score_windows(topic_text, win_strs)
                best_idx   = scores.index(max(scores))
                best_elig  = wins[best_idx]
                best_score = float(max(scores))
                key = (tid, nct)
                maxp_windows[key] = best_elig
                maxp_scores[key]  = best_score
                cf.write(json.dumps({
                    'topic_id': tid, 'nct_id': nct,
                    'elig_window': best_elig, 'maxp_score': best_score
                }) + '\n')

    del mp_model
    torch.cuda.empty_cache()
    import gc; gc.collect()
    print(f'MaxP complete: {len(maxp_windows):,} pairs → {MAXP_CACHE}')

In [ ]:
import random
import numpy as np
from collections import Counter
random.seed(42)

def doc_str(nct, with_desc=False):
    """Full concatenated doc string — used by KZ sanity check in the gate cell."""
    r = fields.get(nct)
    if not r: return ''
    s = f"title: {r['title']} condition: {r['condition']} eligibility: {r['eligibility']}"
    if with_desc: s += f" description: {r['detailed'] or r['summary']}"
    return s[:DOC_CHARS]

def maxp_doc_str(nct, topic_id, with_desc=False):
    """MaxP-selected eligibility window for this (topic, trial) pair."""
    r = fields.get(nct)
    if not r: return ''
    title    = r.get('title', '') or ''
    cond     = r.get('condition', '') or ''
    elig_win = maxp_windows.get((topic_id, nct),
                                (r.get('eligibility', '') or '')[:WINDOW_CHARS])
    s = f"title: {title} condition: {cond} eligibility: {elig_win}"
    if with_desc:
        desc = r.get('detailed') or r.get('summary') or ''
        s += f" description: {desc[:300]}"
    return s[:DOC_CHARS]

def hard_negs_monot5(topic_id, neg_list, k=3):
    """Top-k negatives ranked by base-model MaxP score (highest = hardest = most
    misleading to the base model). This is h2oloo's monoT5_MED hard-negative
    selection: the model's own false positives, not BM25 lexical overlap.

    Falls back to score=0 for any trial missing from the maxp_scores cache
    (shouldn't happen since all judged trials were scored in cell-maxp).
    Returns up to k hard + 1 weak (random from remaining) negatives.
    """
    if not neg_list:
        return []
    scored = sorted(neg_list,
                    key=lambda nct: maxp_scores.get((topic_id, nct), 0.0),
                    reverse=True)
    hard = scored[:k]
    weak_pool = scored[k:]
    if weak_pool:
        hard.append(random.choice(weak_pool))
    return hard

# Score distribution sanity check on the cached scores.
all_scores_sample = [maxp_scores[k] for k in list(maxp_scores)[:200]]
print(f'MaxP score sample (n=200): mean={np.mean(all_scores_sample):.2f}  '
      f'min={np.min(all_scores_sample):.2f}  max={np.max(all_scores_sample):.2f}')

examples = []
for tid, d2r in judg.items():
    pt = topic2text.get(tid)
    if pt is None: continue
    pos = [d for d, r in d2r.items() if r >= 1 and d in fields]
    neg = [d for d, r in d2r.items() if r == 0 and d in fields]
    if not pos or not neg: continue
    for d in pos:
        for wd in (False, True):  # templates (2) and (4)
            examples.append((f'Query: {pt} Document: {maxp_doc_str(d, tid, wd)} Relevant:', 'true'))
        for nd in hard_negs_monot5(tid, neg, k=3):
            examples.append((f'Query: {pt} Document: {maxp_doc_str(nd, tid, random.random() < 0.5)} Relevant:', 'false'))

random.shuffle(examples)
print(f'{len(examples):,} training examples', Counter(t for _, t in examples))

# Spot-check 3 examples to verify MaxP windows are non-empty and meaningful.
print('\nSample training examples:')
for i, (inp, label) in enumerate(random.sample(examples, 3)):
    print(f'  [{label}] {inp[:200]}...')

In [ ]:
# ── Tokenize training examples ───────────────────────────────────────────────
from transformers import T5Tokenizer
from tqdm.auto import tqdm

tok_train = T5Tokenizer.from_pretrained(BASE_MODEL)

tokenized = []
for x, y in tqdm(examples, desc='tokenize', leave=False):
    xi = tok_train(x, truncation=True, max_length=MAX_LEN, return_tensors='pt')
    yi = tok_train(y, return_tensors='pt')
    tokenized.append((xi.input_ids[0], xi.attention_mask[0], yi.input_ids[0]))
print(f'{len(tokenized):,} examples pre-tokenized')

In [ ]:
# ── TREC21 canary setup ───────────────────────────────────────────────────────
# 20 pre-tokenized TREC21 pairs (10 topics × 1 pos + 1 neg). Scored every 100
# training steps to detect catastrophic forgetting. Run this cell once per
# session before cell-train — it reuses corpus data from the gate cell if
# already loaded, otherwise fetches it itself (~2 min).
import random as _cr
from ctmatch.experiments import ExperimentConfig, load_corpus, load_eval
_cr.seed(0)

if 'id2fields' not in dir():
    print('Loading TREC21 corpus for canary...')
    _cfg = ExperimentConfig(data_root=DATA_ROOT)
    _cids, _cfs = load_corpus(_cfg)
    id2fields = dict(zip(_cids, _cfs))
    _sets = load_eval(_cfg, ['trec21'])
    rel21 = _sets['trec21']['rel_dict']
    t2t21 = _sets['trec21']['topic2text']
    print('loaded')

def _can_str(df):
    t = df.get('brief_title') or df.get('official_title') or ''
    c = df.get('conditions', '') or ''
    if isinstance(c, list): c = ', '.join(str(x) for x in c if x)
    e = df.get('eligibility', '') or ''
    return f'title: {t} condition: {c} eligibility: {e}'[:DOC_CHARS]

canary_pairs = []
_tids = list(rel21.items()); _cr.shuffle(_tids)
for _tid, _d2r in _tids:
    if len(canary_pairs) >= 20: break
    _qt = t2t21.get(_tid, '')
    if not _qt: continue
    _pos = [d for d, r in _d2r.items() if r >= 2 and d in id2fields]
    _neg = [d for d, r in _d2r.items() if r == 0 and d in id2fields]
    if not _pos or not _neg: continue
    for _d, _lbl in [(_cr.choice(_pos), 'true'), (_cr.choice(_neg), 'false')]:
        _xi = tok_train(f'Query: {_qt} Document: {_can_str(id2fields[_d])} Relevant:',
                        truncation=True, max_length=MAX_LEN, return_tensors='pt')
        canary_pairs.append((_xi.input_ids[0], _xi.attention_mask[0], _lbl))

print(f'TREC21 canary ready: {len(canary_pairs)} pairs ({len(canary_pairs)//2} topics)')

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5ForConditionalGeneration, Adafactor
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from tqdm.auto import tqdm

CKPT_EVERY = 200

class DS(Dataset):
    def __init__(s, ex): s.ex = ex
    def __len__(s): return len(s.ex)
    def __getitem__(s, i): return s.ex[i]

def collate(b):
    from torch.nn.utils.rnn import pad_sequence
    ii = pad_sequence([x[0] for x in b], batch_first=True, padding_value=tok_train.pad_token_id)
    am = pad_sequence([x[1] for x in b], batch_first=True, padding_value=0)
    lb = pad_sequence([x[2] for x in b], batch_first=True, padding_value=-100)
    return ii, am, lb

def canary_check(m, true_id, false_id, init_gap):
    """Score the 20 TREC21 canary pairs; return (gap, gap_pct) or None."""
    if not canary_pairs: return None
    m.eval()
    pos_m, neg_m = [], []
    with torch.no_grad():
        for ii, am, lbl in canary_pairs:
            dec = torch.zeros((1, 1), dtype=torch.long, device=device)
            out = m(input_ids=ii.unsqueeze(0).to(device),
                    attention_mask=am.unsqueeze(0).to(device),
                    decoder_input_ids=dec).logits[0, 0]
            lp = torch.log_softmax(out.float(), dim=-1)
            margin = (lp[true_id] - lp[false_id]).item()
            (pos_m if lbl == 'true' else neg_m).append(margin)
    m.train()
    p, n = sum(pos_m) / max(len(pos_m), 1), sum(neg_m) / max(len(neg_m), 1)
    gap = p - n
    pct = gap / init_gap * 100 if init_gap else 0
    flag = '✓' if pct >= 50 else ('⚠ FORGETTING' if pct >= 20 else '✗ ABORT')
    print(f'  → TREC21 canary  pos={p:.2f}  neg={n:.2f}  gap={gap:.2f}  ({pct:.0f}% of init)  {flag}')
    return gap, pct

if os.path.exists(os.path.join(OUT_DIR, 'config.json')):
    print('Final checkpoint exists — skipping. Delete OUT_DIR to retrain.')
else:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # LoRA config — adapts all T5 attention projections (q, v, k, o).
    # Base weights are frozen; only ~14M adapter params are trained.
    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=['q', 'v', 'k', 'o'],
    )

    # Resume from LoRA adapter checkpoint, or start fresh.
    resume_ckpt, start_step = None, 0
    for s in range(STEPS, 0, -CKPT_EVERY):
        c = f'{OUT_DIR}_step{s}'
        if os.path.exists(os.path.join(c, 'training_state.pt')):
            resume_ckpt, start_step = c, s; break

    base = T5ForConditionalGeneration.from_pretrained(
        BASE_MODEL, torch_dtype=torch.float32).to(device)
    if resume_ckpt:
        model = PeftModel.from_pretrained(base, resume_ckpt, is_trainable=True)
        print(f'Resumed LoRA from step {start_step}')
    else:
        model = get_peft_model(base, lora_cfg)
        model.print_trainable_parameters()

    model.train()
    opt = Adafactor(model.parameters(), lr=LR, scale_parameter=False,
                    relative_step=False, warmup_init=False)
    if resume_ckpt:
        st = torch.load(os.path.join(resume_ckpt, 'training_state.pt'), map_location=device)
        opt.load_state_dict(st['opt'])

    TRUE_ID  = tok_train('true',  add_special_tokens=False).input_ids[0]
    FALSE_ID = tok_train('false', add_special_tokens=False).input_ids[0]

    init_gap = None
    if canary_pairs:
        r = canary_check(model, TRUE_ID, FALSE_ID, 1.0)
        if r: init_gap = r[0]; print(f'Canary baseline gap={init_gap:.2f}')

    ACC = BATCH // MICRO
    dl  = DataLoader(DS(tokenized), batch_size=MICRO, shuffle=True, collate_fn=collate,
                     num_workers=2, pin_memory=True, persistent_workers=True)
    step, it, _abort = start_step, iter(dl), False
    pbar = tqdm(total=STEPS, initial=start_step, desc='monoT5_CT (LoRA)')

    while step < STEPS:
        opt.zero_grad(); tot = 0.0
        for _ in range(ACC):
            try: ii, am, lb = next(it)
            except StopIteration: it = iter(dl); ii, am, lb = next(it)
            loss = model(input_ids=ii.to(device), attention_mask=am.to(device),
                         labels=lb.to(device)).loss / ACC
            loss.backward(); tot += loss.item()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        step += 1; pbar.update(1)
        if step % 20  == 0: print(f'step {step:4d}/{STEPS}  loss={tot:.4f}')
        if step % 50  == 0:
            r = canary_check(model, TRUE_ID, FALSE_ID, init_gap)
            if r and init_gap and r[1] < 20 and step <= 300:
                print(f'  EARLY STOP at step {step} — forgetting confirmed. LoRA should prevent this; check target_modules.')
                _abort = True; break
        if step % CKPT_EVERY == 0:
            ckpt = f'{OUT_DIR}_step{step}'
            model.save_pretrained(ckpt)   # saves LoRA adapter only (~50MB)
            tok_train.save_pretrained(ckpt)
            torch.save({'step': step, 'opt': opt.state_dict()},
                       os.path.join(ckpt, 'training_state.pt'))
            print(f'  → saved {ckpt}')

    pbar.close()
    if not _abort:
        # Merge LoRA adapters into base weights → standard T5 checkpoint.
        # Gate/sweep cells load with T5ForConditionalGeneration, no PEFT needed.
        print('Merging LoRA adapters into base weights...')
        merged = model.merge_and_unload()
        merged.save_pretrained(OUT_DIR); tok_train.save_pretrained(OUT_DIR)
        print('Saved merged monoT5_CT ->', OUT_DIR)
    else:
        print('Training aborted — no final checkpoint written.')

## Diagnostic: the fine-tune fails the gate — is the *base* model doing the work?

The training loop above aborts on the TREC21 canary (forgetting confirmed). This cell confirms the
mechanism by scoring the **base** model (`castorini/monot5-3b-med-msmarco`, pre-fine-tune) on the same
TREC21 judged pool. Base ≈ 0.45 while every fine-tune ≈ 0.20 means the base model's general
medical-relevance knowledge is what ranks TREC21, and KZ fine-tuning overwrites it (transfer failure).
This is the 0.4491 number cited in the header and in `docs/deep_dive_outline.md` §7e.


In [ ]:

# ── Diagnostic: why did TREC21 gate fail despite KZ sanity passing? ──────────
#
# Two candidates:
#   (A) Corpus field coverage: id2fields[d] missing 'conditions'/'eligibility'
#       → model sees "title: X condition:  eligibility: " (empty) for many docs
#   (B) KZ fine-tune overfit: model memorised KZ short queries but can't
#       generalise to TREC21 100-200 word clinical narratives
#
# This cell probes both by:
#   1. Checking what fraction of TREC21 judged docs have empty key fields
#   2. Running the BASE model (castorini/monot5-3b-med-msmarco, pre-fine-tune)
#      on the same TREC21 judged pool
#
# Interpretation:
#   base ≥ 0.50, fine-tuned = 0.20 → KZ fine-tune overfit (Cause B)
#   base ≈ 0.20, fine-tuned = 0.20 → inference/format bug (Cause A or other)
#   base ≥ 0.50, fine-tuned ≥ 0.50 → prev gate run was a fluke / different ckpt

import torch, json, numpy as np
from transformers import T5Tokenizer, T5ForConditionalGeneration
from tqdm.auto import tqdm
from ctmatch.experiments import ExperimentConfig, load_corpus, load_eval, ndcg_at_k

cfg = ExperimentConfig(data_root=DATA_ROOT)
corpus_ids, corpus_fields = load_corpus(cfg)
id2fields = dict(zip(corpus_ids, corpus_fields))
sets = load_eval(cfg, ['trec21'])
rel21 = sets['trec21']['rel_dict']
t2t21 = sets['trec21']['topic2text']

# ── Field coverage check ─────────────────────────────────────────────────────
judged_docs = [d for docs_rel in rel21.values() for d in docs_rel if d in id2fields]
empty_cond  = sum(1 for d in judged_docs if not id2fields[d].get('conditions'))
empty_elig  = sum(1 for d in judged_docs if not id2fields[d].get('eligibility'))
empty_title = sum(1 for d in judged_docs if not (id2fields[d].get('brief_title') or
                                                   id2fields[d].get('official_title')))
print(f'TREC21 judged docs in corpus: {len(judged_docs)}')
print(f'  empty conditions : {empty_cond}/{len(judged_docs)} = {empty_cond/len(judged_docs):.1%}')
print(f'  empty eligibility: {empty_elig}/{len(judged_docs)} = {empty_elig/len(judged_docs):.1%}')
print(f'  empty title      : {empty_title}/{len(judged_docs)} = {empty_title/len(judged_docs):.1%}')

# Show a few val_doc_str samples (the strings the model actually sees)
def val_doc_str(d_fields, with_desc=False):
    title = d_fields.get('brief_title') or d_fields.get('official_title') or ''
    cond  = d_fields.get('conditions', '') or ''
    if isinstance(cond, list): cond = ', '.join(str(c) for c in cond if c)
    elig  = d_fields.get('eligibility', '') or ''
    s = f'title: {title} condition: {cond} eligibility: {elig}'
    if with_desc:
        desc = d_fields.get('detailed_desc') or d_fields.get('brief_summary') or ''
        s += f' description: {desc}'
    return s[:DOC_CHARS]

print('\nSample val_doc_str (first 3 TREC21 judged docs with eligibility):')
shown = 0
for d in judged_docs:
    if id2fields[d].get('eligibility') and shown < 3:
        s = val_doc_str(id2fields[d])
        print(f'  [{d}] {s[:180]}...')
        shown += 1

print('\nSample val_doc_str (first 3 TREC21 judged docs WITHOUT eligibility):')
shown = 0
for d in judged_docs:
    if not id2fields[d].get('eligibility') and shown < 3:
        s = val_doc_str(id2fields[d])
        print(f'  [{d}] {s[:180]}...')
        shown += 1

# ── Base model TREC21 eval ───────────────────────────────────────────────────
print(f'\n── Loading BASE model ({BASE_MODEL}) for TREC21 eval ──')
base_tok   = T5Tokenizer.from_pretrained(BASE_MODEL)
base_model = T5ForConditionalGeneration.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map='auto').eval()

B_TRUE  = base_tok('true',  add_special_tokens=False).input_ids[0]
B_FALSE = base_tok('false', add_special_tokens=False).input_ids[0]

def base_scores(topic_text, doc_list, batch=8):
    inputs = [f'Query: {topic_text} Document: {d} Relevant:' for d in doc_list]
    scores = []
    for i in range(0, len(inputs), batch):
        enc = base_tok(inputs[i:i+batch], return_tensors='pt', padding=True,
                       truncation=True, max_length=MAX_LEN).to(base_model.device)
        dec = torch.zeros((enc['input_ids'].shape[0], 1), dtype=torch.long,
                          device=base_model.device)
        with torch.no_grad():
            logits = base_model(**enc, decoder_input_ids=dec).logits[:, 0, :]
        lp = torch.log_softmax(logits.float(), dim=-1)
        scores.extend((lp[:, B_TRUE] - lp[:, B_FALSE]).cpu().tolist())
    return scores

ndcgs_base = []
for tid, docs_rel in tqdm(rel21.items(), desc='BASE model TREC21'):
    topic_text = t2t21.get(tid, '')
    if not topic_text: continue
    docs = [d for d in docs_rel if d in id2fields]
    if not docs: continue
    doc_strs = [val_doc_str(id2fields[d]) for d in docs]
    sc = base_scores(topic_text, doc_strs, batch=8)
    ranked = [d for _, d in sorted(zip(sc, docs), reverse=True)]
    ndcgs_base.append(ndcg_at_k(ranked, docs_rel))

base_result = float(np.mean(ndcgs_base))
print(f'\nBASE model ({BASE_MODEL}) TREC21 NDCG@10 = {base_result:.4f}')
print()
print('→ The base model carries the TREC21 signal; KZ fine-tuning overwrites it (transfer failure, §7e).')
print('→ Decision: use the base model as the (optional) monoT5 ensemble feature; do not fine-tune on KZ.')

del base_model
torch.cuda.empty_cache()
print('Base model unloaded.')
